In [ ]:
import glob
import numpy as np
import os
import sys

# add parent folder (production) to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.LoRa import MultiBAMv3

import importlib
import utils.my_lora_utils
importlib.reload(utils.my_lora_utils)
from utils.my_lora_utils import *

print(utils.my_lora_utils.__file__)

In [ ]:
dataset_name_folder = "dataset_new_sf9"
input_layer = 256 * 15
layers = [input_layer, 1024, 256]  # compress 3840 → 1024 → 256

GEENRATE_ = False  #### Secure accidently running

################### Load all .npy files ########################################
print("LOAD DATASET")
files = glob.glob(f'{dataset_name_folder}/*.npy')
data_list = [np.load(f) for f in files]
# Flatten each spectrogram to 1D (256*15 = 3840)
X = np.array([d.flatten() for d in data_list])  # shape: (num_samples, 3840)
print(X.shape)
################### Load all .npy files ########################################

multi_bam = MultiBAMv3(layers_dims=layers, eta=1e-5)

if (GEENRATE_):
    
    folder_path = "weight"

    # Check if folder exists, if not create it
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"Folder created: {folder_path}")
    else:
        print(f"Folder already exists: {folder_path}")
   
    layer_losses = multi_bam.train(X, num_epochs=10, batch_size=64)
    
    for i, bam in enumerate(multi_bam.bams):
        np.save(f"{folder_path}/weights_layer_{i}.npy", bam.W)

def load_weight(): 
    ## HOW TO LOAD WEIGHT
    layers = [256*15, 1024, 256] # <-- must match training

    multi_bam = MultiBAMv3(layers_dims=layers, eta=1e-5)
    for i, bam in enumerate(multi_bam.bams):
        bam.W = np.load(f"weight/weights_layer_{i}.npy")
